In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from scipy.stats import norm
import utilities as utils

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GaussianNetwork(nn.Module):
    """
    Neural network mapping x -> parameters of an N_y-dimensional Gaussian.

    Outputs
    -------
    mean : Tensor, shape (..., N_y)
        Gaussian mean.

    scale_tril : Tensor, shape (..., N_y, N_y)
        Lower-triangular Cholesky factor L such that
            covariance = L @ L.T

    covariance : Tensor, shape (..., N_y, N_y)
        Gaussian covariance matrix.
    """

    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_dims=(128, 128),
        min_std=1e-4,
    ):
        super().__init__()

        self.output_dim = output_dim
        self.min_std = min_std

        # Shared feature-processing network
        layers = []
        d = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(d, hidden_dim),
                nn.ReLU(),
            ])
            d = hidden_dim

        self.backbone = nn.Sequential(*layers)

        # Mean requires N_y parameters
        self.mean_head = nn.Linear(d, output_dim)

        # Lower triangular matrix requires
        # N_y * (N_y + 1) / 2 parameters
        n_tril = output_dim * (output_dim + 1) // 2
        self.cholesky_head = nn.Linear(d, n_tril)

        # Indices of lower-triangular matrix elements
        tril_indices = torch.tril_indices(
            row=output_dim,
            col=output_dim,
            offset=0,
        )

        self.register_buffer("tril_indices", tril_indices)


    def forward(self, x):
        h = self.backbone(x)

        # Mean
        mean = self.mean_head(h)

        # Raw parameters for Cholesky factor
        raw_tril = self.cholesky_head(h)

        batch_shape = x.shape[:-1]

        L = torch.zeros(
            *batch_shape,
            self.output_dim,
            self.output_dim,
            device=x.device,
            dtype=x.dtype,
        )

        L[..., self.tril_indices[0], self.tril_indices[1]] = raw_tril

        # Diagonal must be positive.
        diag_idx = torch.arange(self.output_dim, device=x.device)

        raw_diag = L[..., diag_idx, diag_idx]

        L[..., diag_idx, diag_idx] = (
            F.softplus(raw_diag) + self.min_std
        )

        # covariance = L @ L.transpose(-1, -2)

        # return mean, L, covariance
        return mean, L

In [13]:
def generate_trials(n_trials, mirror_trials=False, balanced_starts=False, seed=None):
    """Generate trials from a two-state switching Gaussian process.

    Parameters
    ----------
    n_trials : int
        Number of independent trials to generate.
    balanced_starts : bool, default=False
        If True, exactly half of the trials start in each source
        distribution. This requires an even ``n_trials``.
    seed : int or None, default=None
        Seed for NumPy's random-number generator. Use an integer for
        reproducible simulations.

    Returns
    -------
    X : ndarray of int, shape (n_trials, 12)
        Clipped and rounded observations for each trial.
    Y : ndarray of int, shape (n_trials,)
        Identity of the source for the final draw: 0 denotes the Gaussian
        with mean -17 and 1 denotes the Gaussian with mean 17.
    """
    if isinstance(n_trials, (bool, np.bool_)) or not isinstance(n_trials, (int, np.integer)):
        raise TypeError("n_trials must be an integer")
    if n_trials < 0:
        raise ValueError("n_trials must be non-negative")
    if (balanced_starts or mirror_trials) and n_trials % 2:
        raise ValueError("balanced_starts=True or mirror_trials=True requires an even n_trials")

    rng = np.random.default_rng(seed)
    n_draws = 12
    source_means = np.array([-17.0, 17.0])

    if mirror_trials:
        n_trials_ = n_trials // 2
    else:
        n_trials_ = n_trials

    if balanced_starts:
        starts = np.repeat([0, 1], n_trials_ // 2)
        rng.shuffle(starts)
    else:
        starts = rng.integers(0, 2, size=n_trials_)

    # switches[:, j] indicates whether the source changes between draws
    # j and j + 1. Cumulative parity therefore gives the source at each draw.
    switches = rng.random((n_trials_, n_draws - 1)) < 0.08
    source_paths = np.column_stack(
        [starts, starts[:, None] ^ np.logical_xor.accumulate(switches, axis=1)]
    ).astype(int)

    raw_draws = rng.normal(loc=source_means[source_paths], scale=29.0)
    X = np.rint(np.clip(raw_draws, -90, 90)).astype(int)
    Y = source_paths[:, -1].copy()
    if mirror_trials:
        X = np.concatenate([X, -X], axis=0)
        Y = np.concatenate([Y, 1-Y], axis=0)
    return X, Y

In [14]:
Xsample,Ysample = generate_trials(100000, mirror_trials=True, balanced_starts=True, seed=234)

In [15]:
test_enc = GaussianNetwork(input_dim=12, output_dim=2, hidden_dims=(128, 128), min_std=1e-4)
test_enc.forward(torch.from_numpy(Xsample[:10]).float())[0]

tensor([[-1.4716, -1.0233],
        [ 2.1817,  1.5874],
        [ 0.3365, -1.3914],
        [-1.4280, -0.1708],
        [-0.5508, -0.4011],
        [ 1.2109,  0.4413],
        [-2.6889,  1.0011],
        [ 1.1799, -0.2817],
        [ 2.1070,  0.0917],
        [-0.9275,  0.2085]], grad_fn=<AddmmBackward0>)

In [16]:
def gaussian_kl_divergence_torch(
    true_mean,
    true_scale_tril,
    approx_mean,
    approx_scale_tril,
):
    true_dist = torch.distributions.MultivariateNormal(
        loc=true_mean,
        scale_tril=true_scale_tril,
    )

    approx_dist = torch.distributions.MultivariateNormal(
        loc=approx_mean,
        scale_tril=approx_scale_tril,
    )

    return torch.distributions.kl_divergence(
        true_dist,
        approx_dist,
    )

def p_RgX_mc_est(mean,L,n_samples=1000):
    """Monte Carlo estimate of p(R|X) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_ZgX = torch.distributions.MultivariateNormal(
        loc=mean,
        scale_tril=L,
    )

    ZgX_samples = p_ZgX.rsample((n_samples,))
    RgX_samples = torch.argmax(ZgX_samples, dim=-1)
    RgX_onehot = F.one_hot(RgX_samples, num_classes=mean.shape[-1])
    p_RgX = RgX_onehot.float().mean(dim=0)
    return p_RgX

def p_RgY_mc_est(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of p(R|Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgX = p_RgX_mc_est(mean,L,n_samples=n_samples)
    Y_onehot = F.one_hot(Y, num_classes=mean.shape[-1])
    p_RgY = p_RgX.T @ Y_onehot.float() / Y_onehot.float().sum(dim=0, keepdim=True)

    return p_RgY

def I_RY(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of I(R;Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgY = p_RgY_mc_est(mean,L,Y,n_samples=n_samples)
    p_Y = F.one_hot(Y, num_classes=mean.shape[-1]).float().mean(dim=0,keepdim=True)
    p_R = (p_RgY * p_Y).sum(dim=1, keepdim=True)
    I_RY = (p_RgY * p_Y * (p_RgY / p_R).log()).sum()
    return I_RY

def variational_obj(beta,mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of the variational objective for a Gaussian encoder.

    Parameters
    ----------
    beta : float
        Trade-off parameter between I(X;R) and I(R;Y).

    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    DKL_ZgX = gaussian_kl_divergence_torch(
        true_mean=mean,
        true_scale_tril=L,
        approx_mean=torch.zeros(mean.shape[-1],device=mean.device, dtype=mean.dtype),   # torch.zeros_like(mean, device=mean.device, dtype=mean.dtype),
        approx_scale_tril=torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype)     # torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype).expand(mean.shape[0], -1, -1),
    )

    I_XR_est = DKL_ZgX.mean()
    I_RY_est = I_RY(mean,L,Y,n_samples=n_samples)
    return I_XR_est - beta * I_RY_est

In [17]:
def p_RgX_soft_mc(mean, L, n_samples=128, temperature=0.25):
    """
    Differentiable Monte Carlo approximation to p(R|X).

    Instead of R = argmax(Z), uses softmax(Z / temperature).

    Parameters
    ----------
    mean : Tensor, shape (B, C)
    L : Tensor, shape (B, C, C)
    n_samples : int
    temperature : float
        Lower values more closely approximate argmax.

    Returns
    -------
    p_RgX : Tensor, shape (B, C)
    """

    dist = torch.distributions.MultivariateNormal(
        loc=mean,
        scale_tril=L,
    )

    # Shape: (n_samples, B, C)
    z = dist.rsample((n_samples,))

    # Differentiable approximation to one_hot(argmax(z))
    soft_R = F.softmax(z / temperature, dim=-1)

    # Average over Gaussian samples
    return soft_R.mean(dim=0)

def I_RY_soft(
    mean,
    L,
    Y,
    n_samples=128,
    temperature=0.25,
    eps=1e-8,
):
    """
    Differentiable Monte Carlo estimate of I(R;Y).

    Parameters
    ----------
    mean : Tensor, shape (B, C)
    L : Tensor, shape (B, C, C)
    Y : Tensor, shape (B,)
    """

    C = mean.shape[-1]

    p_RgX = p_RgX_soft_mc(
        mean,
        L,
        n_samples=n_samples,
        temperature=temperature,
    )

    # Shape: (B, C)
    Y_onehot = F.one_hot(Y, num_classes=C).to(mean.dtype)

    # Number / mass of examples in each Y class
    # Shape: (1, C)
    Y_counts = Y_onehot.sum(dim=0, keepdim=True)

    # p(R | Y)
    #
    # p_RgX.T:    (C_R, B)
    # Y_onehot:   (B, C_Y)
    #
    # result:     (C_R, C_Y)
    p_RgY = (
        p_RgX.T @ Y_onehot
        / Y_counts.clamp_min(1.0)
    )

    # Empirical p(Y)
    # Shape: (1, C_Y)
    p_Y = Y_onehot.mean(dim=0, keepdim=True)

    # p(R) = sum_Y p(R|Y)p(Y)
    # Shape: (C_R, 1)
    p_R = (p_RgY * p_Y).sum(dim=1, keepdim=True)

    # Numerically stable mutual information
    p_RgY_safe = p_RgY.clamp_min(eps)
    p_R_safe = p_R.clamp_min(eps)

    I = (
        p_RgY
        * p_Y
        * (
            torch.log(p_RgY_safe)
            - torch.log(p_R_safe)
        )
    ).sum()

    return I

def variational_obj_train(
    beta,
    mean,
    L,
    Y,
    n_samples=128,
    temperature=0.25,
):
    """
    Differentiable training objective:
        E_X KL[p(Z|X) || N(0,I)] - beta * I(R;Y)
    """

    C = mean.shape[-1]

    prior_mean = torch.zeros(
        C,
        device=mean.device,
        dtype=mean.dtype,
    )

    prior_L = torch.eye(
        C,
        device=mean.device,
        dtype=mean.dtype,
    )

    DKL_ZgX = gaussian_kl_divergence_torch(
        true_mean=mean,
        true_scale_tril=L,
        approx_mean=prior_mean,
        approx_scale_tril=prior_L,
    )

    I_XZ_est = DKL_ZgX.mean()

    I_RY_est = I_RY_soft(
        mean,
        L,
        Y,
        n_samples=n_samples,
        temperature=temperature,
    )

    loss = I_XZ_est - beta * I_RY_est

    return loss, I_XZ_est, I_RY_est

In [18]:
X, Y = generate_trials(
    n_trials=10000,
    mirror_trials=True,
    seed=123,
)

X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.long)

# normalize X to be in the range [-1, 1]
X = X / 90.0

In [19]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [20]:
X = X.to(device)
Y = Y.to(device)

model = GaussianNetwork(
    input_dim=12,
    output_dim=2,
    hidden_dims=(128, 128),
).to(device)

In [23]:
def temperature_schedule(
    epoch,
    start=0.5,
    end=0.1,
    decay_epochs=500,
):
    frac = min(epoch / decay_epochs, 1.0)

    return start * (end / start) ** frac

def train_model(
    model,
    X,
    Y,
    beta,
    n_epochs=2000,
    lr=1e-3,
    n_samples=128,
    temperature=0.25,
    temperature_schedule=None,
    grad_clip=10.0,
    print_every=100,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
    )

    history = {
        "loss": [],
        "I_XZ": [],
        "I_RY": [],
    }

    model.train()

    for epoch in range(1, n_epochs + 1):

        # ----------------------------------
        # Forward pass
        # ----------------------------------

        mean, L = model(X)

        loss, I_XZ_est, I_RY_est = variational_obj_train(
            beta=beta,
            mean=mean,
            L=L,
            Y=Y,
            n_samples=n_samples,
            temperature=temperature_schedule(epoch) if temperature_schedule is not None else temperature,
        )

        # ----------------------------------
        # Backward pass
        # ----------------------------------

        optimizer.zero_grad()

        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                grad_clip,
            )

        optimizer.step()

        # ----------------------------------
        # Logging
        # ----------------------------------

        history["loss"].append(loss.item())
        history["I_XZ"].append(I_XZ_est.item())
        history["I_RY"].append(I_RY_est.item())

        if epoch % print_every == 0 or epoch == 1:
            print(
                f"Epoch {epoch:5d} | "
                f"Loss = {loss.item():.5f} | "
                f"I_XZ = {I_XZ_est.item():.5f} | "
                f"I_RY = {I_RY_est.item():.5f}"
            )

    return history

from torch.utils.data import TensorDataset, DataLoader


def train_model_minibatch(
    model,
    X,
    Y,
    beta,
    n_epochs=500,
    batch_size=1024,
    lr=1e-3,
    n_samples=64,
    temperature=0.25,
    temperature_schedule=None,
    grad_clip=10.0,
    print_every=10,
):
    dataset = TensorDataset(X, Y)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
    )

    history = {
        "loss": [],
        "I_XZ": [],
        "I_RY": [],
    }

    for epoch in range(1, n_epochs + 1):

        model.train()

        epoch_loss = 0.0
        epoch_ixr = 0.0
        epoch_iry = 0.0
        n_seen = 0

        for X_batch, Y_batch in loader:

            # If X/Y were originally kept on CPU:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            mean, L = model(X_batch)

            loss, I_XZ_est, I_RY_est = variational_obj_train(
                beta=beta,
                mean=mean,
                L=L,
                Y=Y_batch,
                n_samples=n_samples,
                temperature=temperature_schedule(epoch) if temperature_schedule is not None else temperature,
            )

            optimizer.zero_grad()
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    grad_clip,
                )

            optimizer.step()

            B = X_batch.shape[0]

            epoch_loss += loss.item() * B
            epoch_ixr += I_XZ_est.item() * B
            epoch_iry += I_RY_est.item() * B

            n_seen += B

        epoch_loss /= n_seen
        epoch_ixr /= n_seen
        epoch_iry /= n_seen

        history["loss"].append(epoch_loss)
        history["I_XZ"].append(epoch_ixr)
        history["I_RY"].append(epoch_iry)

        if epoch % print_every == 0 or epoch == 1:
            print(
                f"Epoch {epoch:4d} | "
                f"Loss = {epoch_loss:.5f} | "
                f"I_XZ = {epoch_ixr:.5f} | "
                f"I_RY = {epoch_iry:.5f}"
            )

    return history

In [24]:
# ==========================================
# Generate dataset
# ==========================================

X_np, Y_np = generate_trials(
    n_trials=10000,
    mirror_trials=True,
    seed=123,
)

X = torch.tensor(
    X_np,
    dtype=torch.float32,
) / 90.0

Y = torch.tensor(
    Y_np,
    dtype=torch.long,
)


# ==========================================
# Device
# ==========================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)


# ==========================================
# Model
# ==========================================

model = GaussianNetwork(
    input_dim=12,
    output_dim=2,
    hidden_dims=(128, 128),
).to(device)


# ==========================================
# Train
# ==========================================

history = train_model_minibatch(
    model=model,
    X=X,
    Y=Y,
    beta=5.0,
    n_epochs=500,
    batch_size=1024,
    lr=1e-3,
    n_samples=64,
    temperature=0.25,
)

Using: cuda
Epoch    1 | Loss = 0.17595 | I_XZ = 0.17616 | I_RY = 0.00004
Epoch   10 | Loss = 0.00091 | I_XZ = 0.00276 | I_RY = 0.00037
Epoch   20 | Loss = -0.14589 | I_XZ = 0.45061 | I_RY = 0.11930
Epoch   30 | Loss = -0.21176 | I_XZ = 0.54263 | I_RY = 0.15088
Epoch   40 | Loss = -0.23378 | I_XZ = 0.54834 | I_RY = 0.15642
Epoch   50 | Loss = -0.25439 | I_XZ = 0.57674 | I_RY = 0.16623
Epoch   60 | Loss = -0.27461 | I_XZ = 0.58824 | I_RY = 0.17257
Epoch   70 | Loss = -0.30344 | I_XZ = 0.59705 | I_RY = 0.18010
Epoch   80 | Loss = -0.33832 | I_XZ = 0.63181 | I_RY = 0.19403
Epoch   90 | Loss = -0.36667 | I_XZ = 0.65236 | I_RY = 0.20381
Epoch  100 | Loss = -0.40958 | I_XZ = 0.69607 | I_RY = 0.22113
Epoch  110 | Loss = -0.45349 | I_XZ = 0.71451 | I_RY = 0.23360
Epoch  120 | Loss = -0.49032 | I_XZ = 0.74748 | I_RY = 0.24756
Epoch  130 | Loss = -0.52841 | I_XZ = 0.76677 | I_RY = 0.25904
Epoch  140 | Loss = -0.56860 | I_XZ = 0.79543 | I_RY = 0.27281
Epoch  150 | Loss = -0.60787 | I_XZ = 0.82760

In [13]:
mean = test_enc.forward(torch.from_numpy(Xsample).float())[0]
L = test_enc.forward(torch.from_numpy(Xsample).float())[1]
Y = torch.from_numpy(Ysample).long()
beta = 5.0

In [14]:
DKL_ZgX = gaussian_kl_divergence_torch(
        true_mean=mean,
        true_scale_tril=L,
        approx_mean=torch.zeros_like(mean, device=mean.device, dtype=mean.dtype),
        approx_scale_tril=torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype).expand(mean.shape[0], -1, -1),
    )

In [15]:
DKL_ZgX

tensor([ 5.1153, 21.7811, 19.3255,  ..., 21.3345, 15.9348, 22.1847],
       grad_fn=<AddBackward0>)

In [16]:
variational_obj(beta,mean,L,Y,n_samples=1000)

tensor(25.3336, grad_fn=<SubBackward0>)

In [17]:
variational_obj(1.0,mean,L,Y,n_samples=1000)

tensor(25.3694, grad_fn=<SubBackward0>)

In [18]:
mean = test_enc.forward(torch.from_numpy(Xsample[:10]).float())[0]
L = test_enc.forward(torch.from_numpy(Xsample[:10]).float())[1]

In [19]:
gaussian_kl_divergence_torch(
    true_mean=mean,
    true_scale_tril=L,
    approx_mean=torch.zeros_like(mean, device=mean.device, dtype=mean.dtype),
    approx_scale_tril=torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype).expand(mean.shape[0], -1, -1),
)

tensor([ 5.1153, 21.7811, 19.3254, 16.7718, 39.8435,  7.8887, 19.9293, 40.4803,
         7.4287, 10.0132], grad_fn=<AddBackward0>)

In [20]:
mean = test_enc.forward(torch.from_numpy(Xsample[:10000]).float())[0]
L = test_enc.forward(torch.from_numpy(Xsample[:10000]).float())[1]
Y = torch.from_numpy(Ysample[:10000]).long()

In [21]:
p_RgY = p_RgY_mc_est(mean,L,Y,n_samples=1000)
p_RgY

tensor([[0.5200, 0.6594],
        [0.4800, 0.3406]])

In [22]:
p_Y = F.one_hot(Y, num_classes=mean.shape[-1]).float().mean(dim=0,keepdim=True)
p_Y

tensor([[0.4962, 0.5038]])

In [23]:
p_R = (p_RgY * p_Y).sum(dim=1, keepdim=True)
p_R

tensor([[0.5902],
        [0.4098]])

In [24]:
(p_RgY * p_Y * (p_RgY / p_R).log()).sum(dim=1, keepdim=True).sum()

tensor(0.0101)

In [25]:
(p_RgY * p_Y * (p_RgY / p_R).log()).sum()

tensor(0.0101)

In [26]:
I_RY(mean,L,Y,n_samples=1000)

tensor(0.0101)

In [27]:
def I_RY(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of I(R;Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgY = p_RgY_mc_est(mean,L,Y,n_samples=n_samples)
    p_Y = F.one_hot(Y, num_classes=mean.shape[-1]).float().mean(dim=0,keepdim=True)
    p_R = (p_RgY * p_Y).sum(dim=1, keepdim=True)
    I_RY = (p_RgY * p_Y * (p_RgY / p_R).log()).sum()
    return I_RY

In [28]:
p_RgX = p_RgX_mc_est(mean,L,n_samples=1000)
p_RgX.shape

torch.Size([10000, 2])

In [29]:
p_RgX

tensor([[0.9410, 0.0590],
        [0.5330, 0.4670],
        [0.5470, 0.4530],
        ...,
        [0.7600, 0.2400],
        [0.7520, 0.2480],
        [0.2640, 0.7360]])

In [30]:
Y_onehot = F.one_hot(torch.from_numpy(Ysample[:10]), num_classes=mean.shape[-1])
Y_onehot.shape

torch.Size([10, 2])

In [31]:
p_RgX.T @ Y_onehot.float()

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x10000 and 10x2)

In [48]:
p_RgX.T @ Y_onehot.float() / Y_onehot.float().sum(dim=0, keepdim=True)

tensor([[0.3717, 0.4017],
        [0.6283, 0.5983]])

In [46]:
p_RgX.T

tensor([[0.0830, 0.2530, 0.5740, 0.3660, 0.5270, 0.7120, 0.4610, 0.7260, 0.0370,
         0.1880],
        [0.9170, 0.7470, 0.4260, 0.6340, 0.4730, 0.2880, 0.5390, 0.2740, 0.9630,
         0.8120]])

In [43]:
Y_onehot

tensor([[0, 1],
        [0, 1],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [0, 1],
        [1, 0],
        [0, 1]])

In [47]:
Y_onehot.float().sum(dim=0, keepdim=True)

tensor([[3., 7.]])

In [ ]:
def p_RgY_mc_est(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of p(R|Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgX = p_RgX_mc_est(mean,L,n_samples=n_samples)
    Y_onehot = F.one_hot(Y, num_classes=mean.shape[-1])
    p_RgY = p_RgX.T @ Y_onehot.float() / Y_onehot.float().sum(dim=0, keepdim=True)

    return p_RgY

In [13]:
p_ZgX = torch.distributions.MultivariateNormal(
        loc=mean,
        scale_tril=L,
    )

In [16]:
ZgX_samples = p_ZgX.rsample((1000,))
ZgX_samples.shape

torch.Size([1000, 10, 2])

In [32]:
ZgX_samples[2]

tensor([[ 1.3138,  3.6046],
        [-1.4387,  7.3054],
        [ 1.1912,  1.5843],
        [-1.6487, -2.1961],
        [ 0.7811,  3.2515],
        [ 2.7614,  3.3366],
        [-2.8398, -6.3952],
        [ 3.2114, -6.5034],
        [-2.9531,  2.6532],
        [-2.0501,  3.0222]], grad_fn=<SelectBackward0>)

In [18]:
R_samples = torch.argmax(ZgX_samples, dim=-1)
R_samples.shape

torch.Size([1000, 10])

In [22]:
R_onehot = F.one_hot(R_samples, num_classes=mean.shape[-1])
R_onehot.shape

torch.Size([1000, 10, 2])

In [33]:
R_samples[2]

tensor([1, 1, 1, 0, 1, 1, 0, 0, 1, 1])

In [37]:
R_onehot.float().mean(dim=0)

tensor([[0.0820, 0.9180],
        [0.2740, 0.7260],
        [0.6030, 0.3970],
        [0.3610, 0.6390],
        [0.4920, 0.5080],
        [0.6930, 0.3070],
        [0.4590, 0.5410],
        [0.7290, 0.2710],
        [0.0430, 0.9570],
        [0.1790, 0.8210]])

In [23]:
R_onehot

tensor([[[0, 1],
         [1, 0],
         [1, 0],
         ...,
         [1, 0],
         [0, 1],
         [0, 1]],

        [[1, 0],
         [0, 1],
         [1, 0],
         ...,
         [1, 0],
         [0, 1],
         [0, 1]],

        [[0, 1],
         [0, 1],
         [0, 1],
         ...,
         [1, 0],
         [0, 1],
         [0, 1]],

        ...,

        [[0, 1],
         [0, 1],
         [1, 0],
         ...,
         [1, 0],
         [0, 1],
         [0, 1]],

        [[0, 1],
         [0, 1],
         [0, 1],
         ...,
         [1, 0],
         [0, 1],
         [1, 0]],

        [[0, 1],
         [0, 1],
         [0, 1],
         ...,
         [1, 0],
         [0, 1],
         [1, 0]]])

In [19]:
R_samples

tensor([[1, 0, 0,  ..., 0, 1, 1],
        [0, 1, 0,  ..., 0, 1, 1],
        [1, 1, 1,  ..., 0, 1, 1],
        ...,
        [1, 1, 0,  ..., 0, 1, 1],
        [1, 1, 1,  ..., 0, 1, 0],
        [1, 1, 1,  ..., 0, 1, 0]])

In [12]:
p_RgX_mc_est(mean,L,n_samples=1000)

TypeError: argmax(): argument 'dim' must be int, not tuple

In [ ]:
def p_RgX_mc_est(mean,L,n_samples=1000):
    """Monte Carlo estimate of p(R|X) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_ZgX = torch.distributions.MultivariateNormal(
        loc=mean,
        scale_tril=L,
    )

    ZgX_samples = p_ZgX.rsample((n_samples,))
    RgX_samples = torch.argmax(ZgX_samples, dim=-1)
    RgX_onehot = F.one_hot(RgX_samples, num_classes=mean.shape[-1])
    p_RgX = RgX_onehot.float().mean(dim=-2)
    return p_RgX